# 🔗 Interview Questions: Advanced JOINs & Aggregations
## Mastering Multi-Table Operations & Complex Analytics

### 🎯 Why Advanced JOINs & Aggregations Define Mid-to-Senior Roles

**Basic JOINs and aggregations get you hired as a junior engineer. Advanced patterns get you promoted.** Here's why:

1. **Real Data is Messy** - Multiple sources, missing keys, duplicates, NULLs
2. **Complex Business Logic** - Multi-touch attribution, hierarchies, time-series
3. **Performance at Scale** - Wrong JOIN strategy = hours vs seconds
4. **Data Quality** - Detect duplicates, orphans, mismatches
5. **Advanced Analytics** - Rolling aggregates, cohort analysis, funnels

### 💡 What Separates Mid from Senior Engineers

| Mid-Level Engineer | Senior Engineer |
|-------------------|------------------|
| Writes basic INNER JOINs | Knows when to use CROSS JOIN, SELF JOIN |
| Uses GROUP BY | Uses GROUPING SETS, ROLLUP, CUBE |
| "This JOIN worked" | Explains cardinality and JOIN strategies |
| Counts rows | Calculates running totals, moving averages |
| Joins two tables | Designs multi-table JOIN hierarchies |
| Fixes errors when they occur | Prevents fan-out, Cartesian products |

---

### 📊 Interview Question Coverage (20 Questions)

This module covers **7 advanced domains**:

| Topic | Questions | Why It Matters |
|-------|-----------|----------------|
| **Self JOINs** | 3 | Hierarchies, employee-manager, duplicates |
| **CROSS JOINs & Cartesian Products** | 2 | Date series, combinatorics, test data |
| **Anti-JOINs (NOT EXISTS/NOT IN)** | 3 | Find missing data, exclusions |
| **Multi-Way JOINs** | 3 | Real queries JOIN 5+ tables |
| **Advanced Aggregations** | 4 | ROLLUP, CUBE, GROUPING SETS |
| **Conditional Aggregates** | 3 | SUM(CASE WHEN), filtered aggregates |
| **JOIN Performance & Pitfalls** | 2 | Fan-out, duplicate rows, NULL traps |

---

### 🎓 How to Master This Module

1. **Draw Venn diagrams** - Visualize what each JOIN includes/excludes
2. **Understand cardinality** - 1:1, 1:N, N:M relationships affect results
3. **Test with NULL data** - NULLs break JOINs in unexpected ways
4. **Measure row counts** - Always verify: rows in = rows out?
5. **Explain JOIN order** - Optimizer reorders, but you should understand optimal sequence

### 🏆 Interview Success Tips

✅ **Ask about data relationships** - 1:1? 1:N? N:M? Duplicates possible?
✅ **Mention JOIN cardinality** - "This is a 1:N JOIN, so we expect fan-out"
✅ **Discuss alternatives** - "We could use a subquery instead of this JOIN"
✅ **Validate assumptions** - "Let me check for NULLs in the join key first"
✅ **Optimize for scale** - "With 1B rows, we should filter before JOIN"

⚠️ **Red flags that fail interviews:**
- Not understanding the difference between WHERE and ON in LEFT JOINs
- Accidentally creating Cartesian products (missing JOIN condition)
- Not handling NULL join keys
- Can't explain why a query returns more rows than input tables
- Using NOT IN without checking for NULLs

---

**Ready to master advanced JOINs and aggregations? Let's dive in!** 🚀

## 🔄 Section 1: Self JOINs (3 Questions)

A self JOIN joins a table to itself. Critical for hierarchies, comparisons, and detecting patterns within a single table.

### ❓ Question 1: Employee-Manager Hierarchy (Self JOIN)

**Classic Interview Question:**
> "You have an employees table with columns: employee_id, name, manager_id. Write a query to show each employee with their manager's name. Handle employees without managers (e.g., CEO)."

### ✅ Answer 1: Self JOIN for Hierarchies

#### **Solution:**

```sql
SELECT 
  emp.employee_id,
  emp.name AS employee_name,
  emp.manager_id,
  mgr.name AS manager_name,
  COALESCE(mgr.name, 'No Manager') AS manager_or_ceo
FROM employees emp
LEFT JOIN employees mgr 
  ON emp.manager_id = mgr.employee_id
ORDER BY emp.employee_id;
```

#### **Key Concepts:**

**1. Self JOIN Pattern:**
- Join table to itself using aliases (emp, mgr)
- Join condition links child to parent: `emp.manager_id = mgr.employee_id`
- Use LEFT JOIN to include employees without managers

**2. Why LEFT JOIN?**
```sql
-- ❌ INNER JOIN excludes CEO/top-level employees
FROM employees emp
INNER JOIN employees mgr ON emp.manager_id = mgr.employee_id;
-- CEO row disappears!

-- ✅ LEFT JOIN includes everyone
FROM employees emp
LEFT JOIN employees mgr ON emp.manager_id = mgr.employee_id;
-- CEO row has NULL manager_name
```

**3. Multi-Level Hierarchy:**
```sql
-- Show employee, their manager, and their manager's manager
SELECT 
  emp.name AS employee,
  mgr1.name AS manager,
  mgr2.name AS director,
  mgr3.name AS vp
FROM employees emp
LEFT JOIN employees mgr1 ON emp.manager_id = mgr1.employee_id
LEFT JOIN employees mgr2 ON mgr1.manager_id = mgr2.employee_id
LEFT JOIN employees mgr3 ON mgr2.manager_id = mgr3.employee_id;
```

**Limitation:** Fixed depth! For arbitrary depth, use RECURSIVE CTE (covered in Module 9).

#### **Common Use Cases:**

**1. Manager Report (direct reports)**
```sql
SELECT 
  mgr.name AS manager,
  COUNT(emp.employee_id) AS direct_reports
FROM employees mgr
LEFT JOIN employees emp ON mgr.employee_id = emp.manager_id
GROUP BY mgr.name
ORDER BY direct_reports DESC;
```

**2. Find orphaned records (manager_id doesn't exist)**
```sql
SELECT emp.*
FROM employees emp
LEFT JOIN employees mgr ON emp.manager_id = mgr.employee_id
WHERE emp.manager_id IS NOT NULL  -- Should have a manager
  AND mgr.employee_id IS NULL;    -- But manager doesn't exist
```

**3. Peer comparison (same manager)**
```sql
-- Employees with same manager
SELECT 
  e1.name AS employee1,
  e2.name AS employee2,
  e1.manager_id,
  mgr.name AS shared_manager
FROM employees e1
JOIN employees e2 
  ON e1.manager_id = e2.manager_id
  AND e1.employee_id < e2.employee_id  -- Avoid duplicates
JOIN employees mgr ON e1.manager_id = mgr.employee_id;
```

#### **Interview Follow-Ups:**

**Q: "What if there are cycles in the data? (A manages B, B manages A)"**

**A:** Self JOINs don't detect cycles. For cycle detection:
```sql
-- Find potential cycles (employees who are each other's managers)
SELECT 
  e1.employee_id,
  e1.name,
  e1.manager_id,
  e2.manager_id AS managers_manager
FROM employees e1
JOIN employees e2 ON e1.manager_id = e2.employee_id
WHERE e1.employee_id = e2.manager_id;
```

**Q: "How would you get the full reporting chain to the CEO?"**

**A:** Use RECURSIVE CTE (covered later):
```sql
WITH RECURSIVE hierarchy AS (
  -- Base: start with target employee
  SELECT employee_id, name, manager_id, 1 AS level
  FROM employees
  WHERE employee_id = 42
  
  UNION ALL
  
  -- Recursive: keep adding managers
  SELECT e.employee_id, e.name, e.manager_id, h.level + 1
  FROM employees e
  JOIN hierarchy h ON e.employee_id = h.manager_id
)
SELECT * FROM hierarchy ORDER BY level;
```

In [0]:
%sql
-- Create employees table with hierarchy
CREATE OR REPLACE TABLE workspace.default.employees_hierarchy (
  employee_id INT,
  name STRING,
  title STRING,
  manager_id INT,
  salary DECIMAL(10,2)
);

INSERT INTO workspace.default.employees_hierarchy VALUES
  (1, 'Alice CEO', 'Chief Executive Officer', NULL, 250000),
  (2, 'Bob VP Eng', 'VP Engineering', 1, 180000),
  (3, 'Carol VP Sales', 'VP Sales', 1, 175000),
  (4, 'Dave Manager', 'Engineering Manager', 2, 140000),
  (5, 'Eve Manager', 'Engineering Manager', 2, 135000),
  (6, 'Frank Engineer', 'Senior Engineer', 4, 120000),
  (7, 'Grace Engineer', 'Engineer', 4, 95000),
  (8, 'Henry Engineer', 'Senior Engineer', 5, 118000),
  (9, 'Ivy Sales', 'Sales Manager', 3, 130000),
  (10, 'Jack Sales Rep', 'Sales Rep', 9, 80000);

-- Self JOIN: Show employees with their managers
SELECT 
  emp.employee_id,
  emp.name AS employee_name,
  emp.title AS employee_title,
  COALESCE(mgr.name, 'No Manager (CEO)') AS manager_name,
  COALESCE(mgr.title, 'N/A') AS manager_title
FROM workspace.default.employees_hierarchy emp
LEFT JOIN workspace.default.employees_hierarchy mgr 
  ON emp.manager_id = mgr.employee_id
ORDER BY emp.employee_id;

-- Count direct reports per manager
SELECT 
  mgr.name AS manager,
  mgr.title,
  COUNT(emp.employee_id) AS direct_reports,
  AVG(emp.salary) AS avg_team_salary
FROM workspace.default.employees_hierarchy mgr
LEFT JOIN workspace.default.employees_hierarchy emp 
  ON mgr.employee_id = emp.manager_id
GROUP BY mgr.name, mgr.title
HAVING COUNT(emp.employee_id) > 0
ORDER BY direct_reports DESC;

### ❓ Question 2: Find Duplicate Records (Self JOIN)

**Data Quality Interview Question:**
> "Write a query to find customers with duplicate email addresses. Show all duplicate records, including the count of how many times each email appears."

### ✅ Answer 2: Duplicate Detection Strategies

#### **Method 1: Self JOIN (Shows All Duplicates)**

```sql
SELECT DISTINCT
  c1.customer_id,
  c1.email,
  c1.name,
  c1.created_date
FROM customers c1
JOIN customers c2 
  ON c1.email = c2.email
  AND c1.customer_id != c2.customer_id  -- Different records, same email
ORDER BY c1.email, c1.customer_id;
```

**How it works:**
- Joins table to itself on email
- Excludes self-match with `c1.customer_id != c2.customer_id`
- Every duplicate email appears multiple times

#### **Method 2: GROUP BY + HAVING (Shows Summary)**

```sql
SELECT 
  email,
  COUNT(*) AS duplicate_count,
  COUNT(DISTINCT customer_id) AS unique_customers,
  STRING_AGG(CAST(customer_id AS STRING), ', ') AS customer_ids
FROM customers
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;
```

**Pros:** Fast, shows summary statistics
**Cons:** Doesn't show individual duplicate records

#### **Method 3: Window Function (Best for Deduplication)**

```sql
WITH ranked_customers AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY email 
      ORDER BY created_date DESC, customer_id DESC
    ) AS row_num,
    COUNT(*) OVER (PARTITION BY email) AS dup_count
  FROM customers
)
SELECT 
  customer_id,
  email,
  name,
  created_date,
  row_num,
  dup_count
FROM ranked_customers
WHERE dup_count > 1  -- Only show emails with duplicates
ORDER BY email, row_num;
```

**Pros:** 
- Shows all duplicates with ranking
- Easy to identify which to keep (row_num = 1)
- Can delete duplicates with `WHERE row_num > 1`

#### **Complete Deduplication Pattern:**

```sql
-- Step 1: Identify duplicates to remove
WITH duplicates_to_remove AS (
  SELECT customer_id
  FROM (
    SELECT 
      customer_id,
      ROW_NUMBER() OVER (
        PARTITION BY email
        ORDER BY 
          created_date DESC,        -- Keep newest
          last_activity_date DESC,  -- Or most active
          customer_id DESC          -- Or highest ID
      ) AS rn
    FROM customers
  )
  WHERE rn > 1  -- All but the first
)
-- Step 2: Delete duplicates
DELETE FROM customers
WHERE customer_id IN (SELECT customer_id FROM duplicates_to_remove);

-- Step 3: Add unique constraint to prevent future duplicates
ALTER TABLE customers ADD CONSTRAINT unique_email UNIQUE (email);
```

#### **Multi-Column Duplicates:**

```sql
-- Duplicates based on multiple columns
WITH ranked AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY email, phone_number, address
      ORDER BY created_date DESC
    ) AS rn
  FROM customers
)
SELECT * FROM ranked WHERE rn > 1;
```

#### **Fuzzy Duplicates (Similar but not exact):**

```sql
-- Find customers with similar names (Levenshtein distance)
SELECT 
  c1.customer_id AS id1,
  c1.name AS name1,
  c2.customer_id AS id2,
  c2.name AS name2,
  LEVENSHTEIN(c1.name, c2.name) AS name_distance
FROM customers c1
JOIN customers c2 
  ON c1.customer_id < c2.customer_id  -- Avoid duplicates
  AND LEVENSHTEIN(c1.name, c2.name) <= 3  -- Similar names
WHERE c1.email != c2.email  -- Different emails
ORDER BY name_distance;
```

#### **Performance Comparison:**

| Method | Speed | Shows All Records | Delete-Ready | Best For |
|--------|-------|-------------------|--------------|----------|
| Self JOIN | ⭐⭐ Slow | ✅ Yes | ❌ No | Small datasets |
| GROUP BY | ⭐⭐⭐⭐ Fast | ❌ No (summary) | ❌ No | Reports/counts |
| Window Function | ⭐⭐⭐⭐⭐ Fastest | ✅ Yes | ✅ Yes | Deduplication |

#### **Interview Follow-Ups:**

**Q: "Which duplicate should you keep?"**

**Criteria to consider:**
1. **Most recent:** `ORDER BY created_date DESC`
2. **Most complete:** `ORDER BY (CASE WHEN phone IS NOT NULL THEN 1 ELSE 0 END + CASE WHEN address IS NOT NULL THEN 1 ELSE 0 END) DESC`
3. **Most activity:** `ORDER BY last_purchase_date DESC`
4. **Highest ID:** `ORDER BY customer_id DESC` (last inserted)

**Q: "How would you prevent duplicates in the future?"**

**A:**
1. Add UNIQUE constraint on key columns
2. Implement application-level validation
3. Use MERGE/UPSERT instead of INSERT
4. Add data quality checks in ETL pipeline

In [0]:
%sql
-- Create customers table with duplicates
CREATE OR REPLACE TABLE workspace.default.customers_with_dupes (
  customer_id INT,
  name STRING,
  email STRING,
  phone STRING,
  created_date DATE
);

INSERT INTO workspace.default.customers_with_dupes VALUES
  (1, 'John Smith', 'john@email.com', '555-0101', '2023-01-15'),
  (2, 'Jane Doe', 'jane@email.com', '555-0102', '2023-02-20'),
  (3, 'John Smith', 'john@email.com', '555-0103', '2023-03-10'),  -- Duplicate email
  (4, 'Bob Wilson', 'bob@email.com', '555-0104', '2023-04-05'),
  (5, 'Alice Brown', 'alice@email.com', '555-0105', '2023-05-12'),
  (6, 'Jane Doe', 'jane@email.com', '555-0106', '2023-06-18'),  -- Duplicate email
  (7, 'Charlie Davis', 'charlie@email.com', '555-0107', '2023-07-22');

-- Method 1: Self JOIN - Show all duplicates
SELECT DISTINCT
  c1.customer_id,
  c1.name,
  c1.email,
  c1.created_date
FROM workspace.default.customers_with_dupes c1
JOIN workspace.default.customers_with_dupes c2 
  ON c1.email = c2.email
  AND c1.customer_id != c2.customer_id
ORDER BY c1.email, c1.customer_id;

-- Method 2: GROUP BY - Summary of duplicates
SELECT 
  email,
  COUNT(*) AS duplicate_count,
  MIN(created_date) AS first_created,
  MAX(created_date) AS last_created
FROM workspace.default.customers_with_dupes
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

-- Method 3: Window Function - Best for deduplication
WITH ranked_customers AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY email 
      ORDER BY created_date DESC, customer_id DESC
    ) AS row_num,
    COUNT(*) OVER (PARTITION BY email) AS dup_count
  FROM workspace.default.customers_with_dupes
)
SELECT 
  customer_id,
  name,
  email,
  created_date,
  row_num,
  dup_count,
  CASE WHEN row_num = 1 THEN 'KEEP' ELSE 'DELETE' END AS action
FROM ranked_customers
WHERE dup_count > 1
ORDER BY email, row_num;

## ✖️ Section 2: CROSS JOINs & Cartesian Products (2 Questions)

CROSS JOIN produces every combination of rows from two tables (Cartesian product). Useful for generating test data and date series.

### ❓ Question 3: Generate Date Series (CROSS JOIN)

**Practical Interview Question:**
> "You need to generate a daily sales report showing revenue for every date in January 2024, even dates with zero sales. Write a query using CROSS JOIN to create a complete date series and fill missing dates with zeros."

### ✅ Answer 3: CROSS JOIN for Date Series

#### **Solution:**

```sql
-- Generate date series for January 2024
WITH date_series AS (
  SELECT SEQUENCE(
    DATE('2024-01-01'),
    DATE('2024-01-31'),
    INTERVAL 1 DAY
  ) AS dates
),
date_spine AS (
  SELECT EXPLODE(dates) AS date_value
  FROM date_series
)
SELECT 
  ds.date_value,
  COALESCE(SUM(s.amount), 0) AS total_sales,
  COUNT(s.sale_id) AS transaction_count
FROM date_spine ds
LEFT JOIN sales s 
  ON DATE(s.sale_date) = ds.date_value
GROUP BY ds.date_value
ORDER BY ds.date_value;
```

#### **CROSS JOIN Use Cases:**

**1. Date Series (Calendar Table)**
```sql
-- Generate all dates in a range
WITH RECURSIVE date_range AS (
  SELECT DATE('2024-01-01') AS date_value
  UNION ALL
  SELECT date_value + INTERVAL 1 DAY
  FROM date_range
  WHERE date_value < DATE('2024-12-31')
)
SELECT * FROM date_range;

-- Or using Databricks SEQUENCE
SELECT EXPLODE(
  SEQUENCE(
    DATE('2024-01-01'),
    DATE('2024-12-31'),
    INTERVAL 1 DAY
  )
) AS date_value;
```

**2. Product-Date Combinations**
```sql
-- Every product for every date (inventory tracking)
SELECT 
  p.product_id,
  p.product_name,
  d.date_value,
  COALESCE(i.quantity_on_hand, 0) AS inventory
FROM products p
CROSS JOIN date_dimension d
LEFT JOIN inventory i 
  ON p.product_id = i.product_id 
  AND d.date_value = i.snapshot_date
WHERE d.date_value BETWEEN '2024-01-01' AND '2024-01-31';
```

**3. All Combinations (Test Data)**
```sql
-- Generate test data: all color-size combinations
WITH colors AS (
  SELECT 'Red' AS color UNION ALL
  SELECT 'Blue' UNION ALL
  SELECT 'Green'
),
sizes AS (
  SELECT 'Small' AS size UNION ALL
  SELECT 'Medium' UNION ALL
  SELECT 'Large'
)
SELECT 
  c.color,
  s.size,
  CONCAT(c.color, ' - ', s.size) AS sku
FROM colors c
CROSS JOIN sizes s;
-- Result: 9 rows (3 colors × 3 sizes)
```

**4. Matrix Queries**
```sql
-- Compare every employee pair
SELECT 
  e1.name AS employee1,
  e2.name AS employee2,
  ABS(e1.salary - e2.salary) AS salary_difference
FROM employees e1
CROSS JOIN employees e2
WHERE e1.employee_id < e2.employee_id  -- Avoid duplicates
ORDER BY salary_difference DESC;
```

#### **CROSS JOIN vs Cartesian Product:**

**Explicit CROSS JOIN (Recommended):**
```sql
SELECT * FROM table1 CROSS JOIN table2;
```

**Implicit Cartesian Product (Avoid):**
```sql
-- Missing JOIN condition = accidental Cartesian product!
SELECT * FROM table1, table2;
-- If table1 has 1000 rows and table2 has 1000 rows = 1,000,000 result rows!
```

#### **Performance Warning:**

⚠️ **CROSS JOIN creates N × M rows**

```
table1: 10,000 rows
table2: 1,000 rows
CROSS JOIN result: 10,000,000 rows!
```

**When to use:**
- Small dimensions (dates, categories, configurations)
- Intentional combinatorics (test data, all possibilities)
- With aggressive filtering after CROSS JOIN

**When NOT to use:**
- Large fact tables
- Without filtering
- Accidentally (missing JOIN condition)

#### **Filling Gaps with CROSS JOIN:**

```sql
-- Find missing dates in time series
WITH date_spine AS (
  SELECT EXPLODE(
    SEQUENCE(
      (SELECT MIN(order_date) FROM orders),
      (SELECT MAX(order_date) FROM orders),
      INTERVAL 1 DAY
    )
  ) AS date_value
)
SELECT ds.date_value AS missing_date
FROM date_spine ds
LEFT JOIN orders o ON ds.date_value = o.order_date
WHERE o.order_date IS NULL;
```

In [0]:
%sql
-- Create sales table with gaps in dates
CREATE OR REPLACE TABLE workspace.default.daily_sales (
  sale_id INT,
  sale_date DATE,
  amount DECIMAL(10,2)
);

INSERT INTO workspace.default.daily_sales VALUES
  (1, '2024-01-01', 1250.00),
  (2, '2024-01-01', 830.50),
  (3, '2024-01-03', 2100.00),  -- Jan 2 missing
  (4, '2024-01-05', 1575.25),  -- Jan 4 missing
  (5, '2024-01-05', 920.00),
  (6, '2024-01-08', 1840.75);  -- Jan 6-7 missing

-- Generate complete date series for January 2024
WITH date_series AS (
  SELECT EXPLODE(
    SEQUENCE(
      DATE('2024-01-01'),
      DATE('2024-01-10'),
      INTERVAL 1 DAY
    )
  ) AS date_value
)
SELECT 
  ds.date_value,
  DAYNAME(ds.date_value) AS day_name,
  COALESCE(SUM(s.amount), 0) AS total_sales,
  COUNT(s.sale_id) AS transaction_count,
  CASE 
    WHEN COUNT(s.sale_id) = 0 THEN 'No Sales'
    ELSE 'Has Sales'
  END AS status
FROM date_series ds
LEFT JOIN workspace.default.daily_sales s 
  ON ds.date_value = s.sale_date
GROUP BY ds.date_value
ORDER BY ds.date_value;

-- CROSS JOIN example: Product-Date combinations
CREATE OR REPLACE TEMP VIEW products_small AS
SELECT 'Laptop' AS product UNION ALL
SELECT 'Mouse' UNION ALL
SELECT 'Keyboard';

WITH dates AS (
  SELECT EXPLODE(
    SEQUENCE(DATE('2024-01-01'), DATE('2024-01-05'), INTERVAL 1 DAY)
  ) AS date_value
)
SELECT 
  p.product,
  d.date_value,
  CONCAT(p.product, ' - ', d.date_value) AS tracking_key
FROM products_small p
CROSS JOIN dates d
ORDER BY p.product, d.date_value;

## ❌ Section 3: Anti-JOINs - NOT EXISTS & NOT IN (3 Questions)

Anti-JOINs find rows in one table that DON'T have matching rows in another. Critical for finding missing data, orphans, and exclusions.

### ❓ Question 4: Find Orphaned Records (Anti-JOIN)

**Data Quality Interview Question:**
> "Find all orders that reference a customer_id that doesn't exist in the customers table. These are 'orphaned' orders caused by data integrity issues."

### ✅ Answer 4: Anti-JOIN Patterns (NOT EXISTS, NOT IN, LEFT JOIN)

#### **Three Methods to Find Non-Matches:**

**Method 1: NOT EXISTS (Recommended)**
```sql
SELECT o.*
FROM orders o
WHERE NOT EXISTS (
  SELECT 1
  FROM customers c
  WHERE c.customer_id = o.customer_id
);
```

**Method 2: NOT IN (Dangerous with NULLs!)**
```sql
SELECT o.*
FROM orders o
WHERE o.customer_id NOT IN (
  SELECT customer_id 
  FROM customers
  WHERE customer_id IS NOT NULL  -- CRITICAL!
);
```

**Method 3: LEFT JOIN + NULL check**
```sql
SELECT o.*
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;  -- No match in customers table
```

#### **Performance & Correctness Comparison:**

| Method | Performance | NULL-Safe | Readability |
|--------|-------------|-----------|-------------|
| NOT EXISTS | ⭐⭐⭐⭐⭐ Fastest | ✅ Yes | ⭐⭐⭐ Good |
| NOT IN | ⭐⭐⭐ Slow | ❌ **NO!** | ⭐⭐⭐⭐⭐ Excellent |
| LEFT JOIN + NULL | ⭐⭐⭐⭐ Fast | ✅ Yes | ⭐⭐⭐⭐ Very Good |

#### **The NOT IN NULL Trap:**

⚠️ **CRITICAL: NOT IN returns NO rows if subquery contains NULL!**

```sql
-- Example of the NULL trap
CREATE TABLE customers (customer_id INT);
INSERT INTO customers VALUES (1), (2), (NULL);  -- One NULL

CREATE TABLE orders (order_id INT, customer_id INT);
INSERT INTO orders VALUES (100, 1), (101, 999);  -- 999 doesn't exist

-- ❌ This returns ZERO rows (wrong!)
SELECT *
FROM orders
WHERE customer_id NOT IN (SELECT customer_id FROM customers);
-- NULL in subquery makes NOT IN always FALSE!

-- ✅ Fix: Filter NULLs
SELECT *
FROM orders
WHERE customer_id NOT IN (
  SELECT customer_id FROM customers WHERE customer_id IS NOT NULL
);
-- Returns order 101 (correct)

-- ✅ Better: Use NOT EXISTS (NULL-safe)
SELECT *
FROM orders o
WHERE NOT EXISTS (
  SELECT 1 FROM customers c WHERE c.customer_id = o.customer_id
);
-- Returns order 101 (correct)
```

**Why NOT IN fails with NULL:**
```
NOT IN (1, 2, NULL) means:
  customer_id != 1 AND customer_id != 2 AND customer_id != NULL
  
Since "customer_id != NULL" is NULL (not TRUE or FALSE),
the entire AND expression becomes NULL, which WHERE treats as FALSE.
```

#### **Common Anti-JOIN Use Cases:**

**1. Find Customers Without Orders**
```sql
-- Method 1: NOT EXISTS
SELECT c.*
FROM customers c
WHERE NOT EXISTS (
  SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
);

-- Method 2: LEFT JOIN
SELECT c.*
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
```

**2. Find Products Never Ordered**
```sql
SELECT p.product_id, p.product_name
FROM products p
WHERE NOT EXISTS (
  SELECT 1 FROM order_items oi WHERE oi.product_id = p.product_id
);
```

**3. Find Missing Dates**
```sql
WITH date_spine AS (
  SELECT EXPLODE(
    SEQUENCE(DATE('2024-01-01'), DATE('2024-01-31'), INTERVAL 1 DAY)
  ) AS date_value
)
SELECT ds.date_value
FROM date_spine ds
WHERE NOT EXISTS (
  SELECT 1 FROM sales s WHERE DATE(s.sale_date) = ds.date_value
);
```

**4. Exclude Specific Values**
```sql
-- Find users who never clicked on ad campaign 'SUMMER2024'
SELECT u.user_id, u.name
FROM users u
WHERE NOT EXISTS (
  SELECT 1 
  FROM ad_clicks ac 
  WHERE ac.user_id = u.user_id 
    AND ac.campaign_id = 'SUMMER2024'
);
```

#### **Exists vs IN (When Both Work):**

**Use EXISTS when:**
- Correlated subquery (references outer query)
- Large subquery result sets
- Subquery may contain NULLs

**Use IN when:**
- Small, static list of values
- Subquery returns small result set
- Values are guaranteed non-NULL

```sql
-- IN: Good for small lists
WHERE status IN ('active', 'pending', 'approved')

-- EXISTS: Better for subqueries
WHERE EXISTS (
  SELECT 1 FROM large_table WHERE ...
)
```

#### **Interview Follow-Up:**

**Q: "Why is NOT EXISTS faster than NOT IN?"**

**A:**
1. **Short-circuit evaluation:** NOT EXISTS stops at first match (doesn't need full scan)
2. **No NULL checking:** Doesn't have to check for NULLs in result set
3. **Better query plans:** Optimizer can use semi-join strategies
4. **Correlated execution:** Can use indexes on joined table

**Q: "When would you use LEFT JOIN instead of NOT EXISTS?"**

**A:**
- When you need other columns from the right table (not just existence check)
- When combining with other LEFT JOINs in same query
- When the pattern is clearer for your team's SQL style

In [0]:
%sql
-- Create tables to demonstrate anti-JOIN
CREATE OR REPLACE TABLE workspace.default.customers_main (
  customer_id INT,
  name STRING,
  email STRING
);

CREATE OR REPLACE TABLE workspace.default.orders_main (
  order_id INT,
  customer_id INT,
  order_date DATE,
  amount DECIMAL(10,2)
);

INSERT INTO workspace.default.customers_main VALUES
  (1, 'Alice', 'alice@email.com'),
  (2, 'Bob', 'bob@email.com'),
  (3, 'Carol', 'carol@email.com'),
  (4, 'Dave', 'dave@email.com');

INSERT INTO workspace.default.orders_main VALUES
  (100, 1, '2024-01-15', 1250.00),
  (101, 1, '2024-02-20', 830.50),
  (102, 2, '2024-01-22', 2100.00),
  (103, 999, '2024-03-10', 1575.25);  -- Orphaned: customer 999 doesn't exist

-- Find orphaned orders (orders without valid customer)
-- Method 1: NOT EXISTS (Best)
SELECT 
  o.order_id,
  o.customer_id,
  o.amount,
  'Orphaned Order' AS status
FROM workspace.default.orders_main o
WHERE NOT EXISTS (
  SELECT 1 FROM workspace.default.customers_main c 
  WHERE c.customer_id = o.customer_id
);

-- Method 2: LEFT JOIN + NULL check
SELECT 
  o.order_id,
  o.customer_id,
  o.amount,
  'Orphaned Order' AS status
FROM workspace.default.orders_main o
LEFT JOIN workspace.default.customers_main c 
  ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

-- Find customers without orders
SELECT 
  c.customer_id,
  c.name,
  c.email,
  'No Orders' AS status
FROM workspace.default.customers_main c
WHERE NOT EXISTS (
  SELECT 1 FROM workspace.default.orders_main o 
  WHERE o.customer_id = c.customer_id
);

## 📊 Section 4: Advanced Aggregations - ROLLUP, CUBE, GROUPING SETS (4 Questions)

Advanced aggregation functions create subtotals and grand totals in a single query. Essential for reporting and analytics.

### ❓ Question 5: Create Hierarchical Subtotals with ROLLUP

**Reporting Interview Question:**
> "Write a query to show sales totals by region and product category, including:
> - Sales per region per category
> - Subtotal per region (all categories)
> - Grand total (all regions, all categories)
> Use ROLLUP to generate all levels in one query."

### ✅ Answer 5: ROLLUP for Hierarchical Aggregations

#### **Solution:**

```sql
SELECT 
  region,
  category,
  SUM(sales_amount) AS total_sales,
  COUNT(*) AS transaction_count
FROM sales
GROUP BY ROLLUP(region, category)
ORDER BY region NULLS LAST, category NULLS LAST;
```

#### **What ROLLUP Does:**

ROLLUP creates **hierarchical subtotals** following the column order:

```
ROLLUP(region, category) generates:
1. GROUP BY region, category      -- Detailed level
2. GROUP BY region                -- Region subtotals
3. Grand total (no GROUP BY)      -- Overall total
```

**Visual Example:**
```
region    | category    | total_sales
----------|-------------|-------------
West      | Electronics | 10,000      ← Detail
West      | Furniture   | 5,000       ← Detail
West      | NULL        | 15,000      ← Region subtotal
East      | Electronics | 8,000       ← Detail
East      | Furniture   | 6,000       ← Detail
East      | NULL        | 14,000      ← Region subtotal
NULL      | NULL        | 29,000      ← Grand total
```

#### **Identifying Subtotal Rows:**

Use **GROUPING()** function to detect NULL from ROLLUP (vs real NULL):

```sql
SELECT 
  region,
  category,
  SUM(sales_amount) AS total_sales,
  GROUPING(region) AS is_region_total,
  GROUPING(category) AS is_category_total,
  CASE 
    WHEN GROUPING(region) = 1 AND GROUPING(category) = 1 THEN 'Grand Total'
    WHEN GROUPING(category) = 1 THEN CONCAT(region, ' Subtotal')
    ELSE 'Detail'
  END AS aggregation_level
FROM sales
GROUP BY ROLLUP(region, category);
```

**GROUPING() returns:**
- `0` = Real group value
- `1` = NULL from ROLLUP/CUBE (aggregation level)

#### **ROLLUP with Multiple Columns:**

```sql
-- Three-level hierarchy
GROUP BY ROLLUP(year, quarter, month)

Generates:
1. GROUP BY year, quarter, month   -- Monthly detail
2. GROUP BY year, quarter          -- Quarterly subtotal
3. GROUP BY year                   -- Yearly subtotal
4. Grand total                     -- Overall
```

#### **Partial ROLLUP:**

```sql
-- ROLLUP only some columns
SELECT 
  region,
  year,
  category,
  SUM(sales) AS total_sales
FROM sales
GROUP BY region, ROLLUP(year, category);

-- Creates subtotals within each region, but not across regions
```

#### **ROLLUP vs Manual UNION:**

**Without ROLLUP (Manual, Inefficient):**
```sql
-- Detail level
SELECT region, category, SUM(sales) FROM sales GROUP BY region, category
UNION ALL
-- Region subtotals
SELECT region, NULL, SUM(sales) FROM sales GROUP BY region
UNION ALL
-- Grand total
SELECT NULL, NULL, SUM(sales) FROM sales;

-- ❌ Scans table 3 times!
```

**With ROLLUP (Efficient):**
```sql
SELECT region, category, SUM(sales)
FROM sales
GROUP BY ROLLUP(region, category);

-- ✅ Scans table once!
```

#### **Real-World Use Cases:**

**1. Sales Report with Subtotals**
```sql
SELECT 
  COALESCE(region, 'ALL REGIONS') AS region,
  COALESCE(product_category, 'ALL CATEGORIES') AS category,
  SUM(revenue) AS total_revenue,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM sales
GROUP BY ROLLUP(region, product_category);
```

**2. Time-Series with Period Totals**
```sql
SELECT 
  year,
  quarter,
  month,
  SUM(orders) AS total_orders
FROM order_summary
GROUP BY ROLLUP(year, quarter, month);
```

**3. Multi-Dimensional Analysis**
```sql
SELECT 
  store_id,
  department,
  product_line,
  SUM(sales_amount) AS total_sales
FROM retail_transactions
GROUP BY ROLLUP(store_id, department, product_line);
```

### ❓ Question 6: Generate All Possible Subtotals with CUBE

**Advanced Analytics Question:**
> "Create a multi-dimensional sales report showing totals for every possible combination of region and product category. Use CUBE to generate all subtotal combinations."

### ✅ Answer 6: CUBE for Multi-Dimensional Aggregations

#### **Solution:**

```sql
SELECT 
  region,
  category,
  SUM(sales_amount) AS total_sales
FROM sales
GROUP BY CUBE(region, category)
ORDER BY region NULLS LAST, category NULLS LAST;
```

#### **What CUBE Does:**

CUBE creates **all possible combinations** of subtotals:

```
CUBE(region, category) generates:
1. GROUP BY region, category      -- Both dimensions
2. GROUP BY region                -- Region only
3. GROUP BY category              -- Category only
4. Grand total (no GROUP BY)      -- Neither dimension
```

**CUBE vs ROLLUP:**

| ROLLUP | CUBE |
|--------|------|
| Hierarchical (follows order) | All combinations |
| N+1 groupings | 2^N groupings |
| `ROLLUP(A, B)` = 3 levels | `CUBE(A, B)` = 4 levels |
| `ROLLUP(A, B, C)` = 4 levels | `CUBE(A, B, C)` = 8 levels |

**Visual Comparison:**

```
ROLLUP(region, category):          CUBE(region, category):
region    category                 region    category
------    --------                 ------    --------
West      Electronics              West      Electronics
West      Furniture                West      Furniture
West      NULL          ←          West      NULL
East      Electronics              East      Electronics
East      Furniture                East      Furniture
East      NULL          ←          East      NULL
NULL      NULL          ← Grand   NULL      Electronics  ← NEW!
NULL      Furniture     ← NEW!
NULL      NULL          ← Grand
```

#### **CUBE with 3 Dimensions:**

```sql
GROUP BY CUBE(region, category, year)

Generates 2^3 = 8 groupings:
1. region, category, year
2. region, category
3. region, year
4. category, year
5. region
6. category
7. year
8. Grand total
```

#### **Using GROUPING_ID() for Complex Logic:**

```sql
SELECT 
  region,
  category,
  year,
  SUM(sales) AS total_sales,
  GROUPING_ID(region, category, year) AS group_id,
  CASE GROUPING_ID(region, category, year)
    WHEN 0 THEN 'Detail (R+C+Y)'
    WHEN 1 THEN 'By Region and Category'
    WHEN 2 THEN 'By Region and Year'
    WHEN 3 THEN 'By Region Only'
    WHEN 4 THEN 'By Category and Year'
    WHEN 5 THEN 'By Category Only'
    WHEN 6 THEN 'By Year Only'
    WHEN 7 THEN 'Grand Total'
  END AS level_description
FROM sales
GROUP BY CUBE(region, category, year);
```

**GROUPING_ID()** returns a bitmask:
```
GROUPING_ID(A, B, C) = 4*GROUPING(A) + 2*GROUPING(B) + 1*GROUPING(C)
```

#### **GROUPING SETS (Custom Combinations):**

```sql
-- Only specific groupings (more efficient than CUBE)
SELECT 
  region,
  category,
  SUM(sales) AS total_sales
FROM sales
GROUP BY GROUPING SETS (
  (region, category),  -- Detail
  (region),            -- Region subtotal
  (category),          -- Category subtotal
  ()                   -- Grand total
);

-- Equivalent to CUBE(region, category) but explicitly defined
```

**When to use GROUPING SETS:**
- You don't need all CUBE combinations
- Performance matters (fewer groupings = faster)
- Business logic requires specific subtotals only

**Example: Only specific combinations**
```sql
GROUP BY GROUPING SETS (
  (year, quarter, month),  -- Monthly detail
  (year, quarter),         -- Quarterly
  (year),                  -- Yearly
  ()                       -- Grand total
)
-- Skips month-only, quarter-only subtotals
```

#### **Performance Considerations:**

**Number of Groupings:**
```
ROLLUP(a, b, c)      = 4 groupings
CUBE(a, b, c)        = 8 groupings
CUBE(a, b, c, d)     = 16 groupings
CUBE(a, b, c, d, e)  = 32 groupings  ⚠️ Expensive!
```

**Optimization Tips:**
1. Use GROUPING SETS for custom subtotals (faster than CUBE)
2. Filter data BEFORE GROUP BY
3. Use materialized views for frequently-accessed subtotals
4. Consider pre-aggregated summary tables for large data

#### **Real-World Example: Sales Dashboard**

```sql
SELECT 
  COALESCE(region, 'All Regions') AS region,
  COALESCE(category, 'All Categories') AS category,
  COALESCE(TO_CHAR(year), 'All Years') AS year,
  SUM(revenue) AS total_revenue,
  COUNT(DISTINCT order_id) AS order_count,
  AVG(order_value) AS avg_order_value,
  GROUPING_ID(region, category, year) AS grouping_level
FROM sales_fact
GROUP BY CUBE(region, category, year)
HAVING GROUPING_ID(region, category, year) IN (0, 3, 5, 7)
  -- Filter to only: detail, region-only, category-only, grand total
ORDER BY 
  GROUPING_ID(region, category, year),
  region NULLS LAST,
  category NULLS LAST,
  year NULLS LAST;
```

In [0]:
%sql
-- Create sales data for ROLLUP/CUBE demos
CREATE OR REPLACE TABLE workspace.default.sales_data (
  sale_id INT,
  region STRING,
  category STRING,
  year INT,
  sales_amount DECIMAL(10,2)
);

INSERT INTO workspace.default.sales_data VALUES
  (1, 'West', 'Electronics', 2023, 10000),
  (2, 'West', 'Electronics', 2023, 8000),
  (3, 'West', 'Furniture', 2023, 5000),
  (4, 'East', 'Electronics', 2023, 12000),
  (5, 'East', 'Furniture', 2023, 6000),
  (6, 'West', 'Electronics', 2024, 15000),
  (7, 'West', 'Furniture', 2024, 7000),
  (8, 'East', 'Electronics', 2024, 13000),
  (9, 'East', 'Furniture', 2024, 8000);

-- ROLLUP: Hierarchical subtotals
SELECT 
  COALESCE(region, 'ALL REGIONS') AS region,
  COALESCE(category, 'ALL CATEGORIES') AS category,
  SUM(sales_amount) AS total_sales,
  COUNT(*) AS transaction_count,
  GROUPING(region) AS is_region_total,
  GROUPING(category) AS is_category_total,
  CASE 
    WHEN GROUPING(region) = 1 AND GROUPING(category) = 1 THEN 'Grand Total'
    WHEN GROUPING(category) = 1 THEN 'Region Subtotal'
    ELSE 'Detail'
  END AS level
FROM workspace.default.sales_data
GROUP BY ROLLUP(region, category)
ORDER BY region, category;

-- CUBE: All combinations of subtotals
SELECT 
  COALESCE(region, 'ALL') AS region,
  COALESCE(category, 'ALL') AS category,
  COALESCE(CAST(year AS STRING), 'ALL') AS year,
  SUM(sales_amount) AS total_sales,
  GROUPING_ID(region, category, year) AS group_id,
  CASE GROUPING_ID(region, category, year)
    WHEN 0 THEN 'Detail: R+C+Y'
    WHEN 1 THEN 'By R+C'
    WHEN 2 THEN 'By R+Y'
    WHEN 3 THEN 'By Region Only'
    WHEN 4 THEN 'By C+Y'
    WHEN 5 THEN 'By Category Only'
    WHEN 6 THEN 'By Year Only'
    WHEN 7 THEN 'Grand Total'
  END AS aggregation_level
FROM workspace.default.sales_data
GROUP BY CUBE(region, category, year)
ORDER BY group_id, region, category, year;

-- GROUPING SETS: Custom combinations
SELECT 
  COALESCE(region, 'ALL') AS region,
  COALESCE(category, 'ALL') AS category,
  SUM(sales_amount) AS total_sales
FROM workspace.default.sales_data
GROUP BY GROUPING SETS (
  (region, category),  -- Detail
  (region),            -- By region
  (category),          -- By category
  ()                   -- Grand total
)
ORDER BY region, category;

## 🎯 Section 5: Conditional Aggregates (3 Questions)

Conditional aggregates use CASE WHEN inside aggregate functions to create pivot-like reports and conditional metrics.

### ❓ Question 7: Create Pivot Report with Conditional Aggregates

**Analytics Interview Question:**
> "Create a monthly sales report showing revenue by product category as columns (one column per category). Use conditional aggregates, not PIVOT syntax."

### ✅ Answer 7: Conditional Aggregates with CASE WHEN

#### **Solution:**

```sql
SELECT 
  DATE_TRUNC('month', order_date) AS month,
  SUM(CASE WHEN category = 'Electronics' THEN amount ELSE 0 END) AS electronics_revenue,
  SUM(CASE WHEN category = 'Furniture' THEN amount ELSE 0 END) AS furniture_revenue,
  SUM(CASE WHEN category = 'Clothing' THEN amount ELSE 0 END) AS clothing_revenue,
  SUM(amount) AS total_revenue
FROM sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month;
```

#### **Key Pattern:**

```sql
SUM(CASE WHEN condition THEN value ELSE 0 END)
```

**How it works:**
1. CASE WHEN includes value if condition is TRUE
2. Returns 0 if condition is FALSE
3. SUM aggregates only the matching values

#### **Common Conditional Aggregate Patterns:**

**1. Count with Condition (Filtered COUNT)**
```sql
SELECT 
  department,
  COUNT(*) AS total_employees,
  COUNT(CASE WHEN salary > 100000 THEN 1 END) AS high_earners,
  COUNT(CASE WHEN hire_date >= '2023-01-01' THEN 1 END) AS recent_hires
FROM employees
GROUP BY department;
```

**Alternative (COUNT with boolean):**
```sql
COUNT(CASE WHEN condition THEN 1 ELSE NULL END)
-- Or simpler:
SUM(CASE WHEN condition THEN 1 ELSE 0 END)
```

**2. Average with Condition**
```sql
SELECT 
  product_id,
  AVG(CASE WHEN rating >= 4 THEN rating END) AS avg_positive_rating,
  AVG(CASE WHEN rating < 4 THEN rating END) AS avg_negative_rating
FROM product_reviews
GROUP BY product_id;
```

**3. Multiple Metrics in One Pass**
```sql
SELECT 
  customer_segment,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_orders,
  SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
  SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending_orders,
  SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed_revenue,
  AVG(CASE WHEN status = 'completed' THEN amount END) AS avg_order_value
FROM orders
GROUP BY customer_segment;
```

**4. Percentage Calculations**
```sql
SELECT 
  region,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN is_active = true THEN 1 ELSE 0 END) AS active_customers,
  ROUND(
    SUM(CASE WHEN is_active = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
    2
  ) AS active_percentage
FROM customers
GROUP BY region;
```

**5. Date Range Buckets**
```sql
SELECT 
  product_id,
  SUM(CASE WHEN order_date >= CURRENT_DATE - INTERVAL 7 DAYS THEN quantity ELSE 0 END) AS last_7_days,
  SUM(CASE WHEN order_date >= CURRENT_DATE - INTERVAL 30 DAYS THEN quantity ELSE 0 END) AS last_30_days,
  SUM(CASE WHEN order_date >= CURRENT_DATE - INTERVAL 90 DAYS THEN quantity ELSE 0 END) AS last_90_days,
  SUM(quantity) AS all_time
FROM order_items
GROUP BY product_id;
```

**6. Cohort Analysis**
```sql
SELECT 
  DATE_TRUNC('month', signup_date) AS cohort_month,
  COUNT(*) AS cohort_size,
  SUM(CASE WHEN last_login >= CURRENT_DATE - INTERVAL 7 DAYS THEN 1 ELSE 0 END) AS active_last_7d,
  SUM(CASE WHEN last_login >= CURRENT_DATE - INTERVAL 30 DAYS THEN 1 ELSE 0 END) AS active_last_30d,
  ROUND(
    SUM(CASE WHEN last_login >= CURRENT_DATE - INTERVAL 30 DAYS THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
    2
  ) AS retention_rate_30d
FROM users
GROUP BY DATE_TRUNC('month', signup_date)
ORDER BY cohort_month;
```

#### **Conditional Aggregates vs PIVOT:**

**Conditional Aggregates (Manual):**
```sql
SELECT 
  month,
  SUM(CASE WHEN category = 'A' THEN sales END) AS cat_a,
  SUM(CASE WHEN category = 'B' THEN sales END) AS cat_b
FROM sales
GROUP BY month;
```

**PIVOT (Automatic):**
```sql
SELECT *
FROM sales
PIVOT (
  SUM(sales)
  FOR category IN ('A' AS cat_a, 'B' AS cat_b)
)
ORDER BY month;
```

**When to use each:**
- **Conditional Aggregates:** More control, works everywhere, can combine with other logic
- **PIVOT:** Cleaner syntax, but less flexible, not all databases support it

#### **Performance Tip:**

```sql
-- ❌ Bad: Multiple passes over data
SELECT 'Electronics' AS category, SUM(amount) FROM sales WHERE category = 'Electronics'
UNION ALL
SELECT 'Furniture', SUM(amount) FROM sales WHERE category = 'Furniture'
UNION ALL
SELECT 'Clothing', SUM(amount) FROM sales WHERE category = 'Clothing';
-- Scans table 3 times!

-- ✅ Good: Single pass with conditional aggregates
SELECT 
  SUM(CASE WHEN category = 'Electronics' THEN amount ELSE 0 END) AS electronics,
  SUM(CASE WHEN category = 'Furniture' THEN amount ELSE 0 END) AS furniture,
  SUM(CASE WHEN category = 'Clothing' THEN amount ELSE 0 END) AS clothing
FROM sales;
-- Scans table once!
```

#### **Advanced: Nested Conditions**

```sql
SELECT 
  customer_id,
  SUM(CASE 
    WHEN order_date >= '2024-01-01' AND status = 'completed' THEN amount
    ELSE 0
  END) AS revenue_2024,
  SUM(CASE 
    WHEN order_date >= '2024-01-01' AND status = 'completed' AND amount > 1000 THEN 1
    ELSE 0
  END) AS high_value_orders_2024
FROM orders
GROUP BY customer_id;
```

#### **Interview Follow-Up:**

**Q: "What's the difference between these two?"**
```sql
-- Version 1
SUM(CASE WHEN category = 'A' THEN amount ELSE 0 END)

-- Version 2
SUM(CASE WHEN category = 'A' THEN amount END)
```

**A:**
- **Version 1:** Returns 0 for all rows (safe, but verbose)
- **Version 2:** Returns NULL for non-matching rows; SUM ignores NULLs (more efficient)
- **Result:** Same total, but Version 2 is slightly more efficient
- **Best Practice:** Use Version 2 for SUM/AVG, Version 1 for clarity or when 0 vs NULL matters

In [0]:
%sql
-- Demo: Conditional Aggregates (Pivot-like reports)

-- Create monthly sales data
CREATE OR REPLACE TABLE workspace.default.monthly_sales (
  sale_id INT,
  sale_date DATE,
  category STRING,
  amount DECIMAL(10,2)
);

INSERT INTO workspace.default.monthly_sales VALUES
  (1, '2024-01-15', 'Electronics', 1200),
  (2, '2024-01-20', 'Furniture', 800),
  (3, '2024-01-25', 'Electronics', 950),
  (4, '2024-02-05', 'Clothing', 450),
  (5, '2024-02-10', 'Electronics', 1500),
  (6, '2024-02-15', 'Furniture', 1100),
  (7, '2024-03-01', 'Clothing', 600),
  (8, '2024-03-10', 'Electronics', 1800),
  (9, '2024-03-20', 'Furniture', 950);

-- Pivot report: Categories as columns
SELECT 
  DATE_TRUNC('month', sale_date) AS month,
  SUM(CASE WHEN category = 'Electronics' THEN amount ELSE 0 END) AS electronics,
  SUM(CASE WHEN category = 'Furniture' THEN amount ELSE 0 END) AS furniture,
  SUM(CASE WHEN category = 'Clothing' THEN amount ELSE 0 END) AS clothing,
  SUM(amount) AS total_revenue,
  COUNT(*) AS total_transactions
FROM workspace.default.monthly_sales
GROUP BY DATE_TRUNC('month', sale_date)
ORDER BY month;

-- Multiple conditional metrics
SELECT 
  category,
  COUNT(*) AS total_sales,
  SUM(CASE WHEN amount > 1000 THEN 1 ELSE 0 END) AS high_value_sales,
  SUM(CASE WHEN amount <= 1000 THEN 1 ELSE 0 END) AS regular_sales,
  AVG(CASE WHEN amount > 1000 THEN amount END) AS avg_high_value,
  AVG(CASE WHEN amount <= 1000 THEN amount END) AS avg_regular,
  ROUND(
    SUM(CASE WHEN amount > 1000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
    2
  ) AS high_value_percentage
FROM workspace.default.monthly_sales
GROUP BY category
ORDER BY total_sales DESC;

## ⚡ Section 6: JOIN Performance & Common Pitfalls (2 Questions)

Understanding JOIN cardinality and avoiding common pitfalls prevents query bugs and performance issues.

### ❓ Question 8: Detect and Fix JOIN Fan-Out

**Debugging Interview Question:**
> "You wrote a query that should return 1000 rows (one per customer), but it's returning 5000 rows. Explain what causes this 'fan-out' problem and how to fix it."

### ✅ Answer 8: JOIN Fan-Out and Cardinality Issues

#### **What is Fan-Out?**

**Fan-out** occurs when a JOIN creates more rows than expected due to:
1. **1:N relationship** (one customer, many orders)
2. **Duplicate keys** in the joined table
3. **Missing GROUP BY** after joining detail tables

#### **Example Problem:**

```sql
-- Expected: 1000 customers
-- Actual: 5000 rows
SELECT 
  c.customer_id,
  c.name,
  o.order_id,
  o.order_date
FROM customers c  -- 1000 rows
JOIN orders o ON c.customer_id = o.customer_id;  -- 5000 orders

-- Result: 5000 rows (one per order, not per customer!)
```

**Why?** Each customer with 5 orders becomes 5 rows.

#### **Fix 1: Add GROUP BY (if you want one row per customer)**

```sql
SELECT 
  c.customer_id,
  c.name,
  COUNT(o.order_id) AS order_count,
  MAX(o.order_date) AS last_order_date
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name;

-- Result: 1000 rows (one per customer)
```

#### **Fix 2: Use Subquery to Aggregate First**

```sql
-- Aggregate orders first, then join
WITH order_summary AS (
  SELECT 
    customer_id,
    COUNT(*) AS order_count,
    MAX(order_date) AS last_order_date
  FROM orders
  GROUP BY customer_id
)
SELECT 
  c.customer_id,
  c.name,
  os.order_count,
  os.last_order_date
FROM customers c
LEFT JOIN order_summary os ON c.customer_id = os.customer_id;

-- Result: 1000 rows, includes customers with 0 orders
```

#### **Detecting Fan-Out:**

**1. Check row counts**
```sql
-- Compare input vs output row counts
WITH input_counts AS (
  SELECT COUNT(*) AS customer_count FROM customers
),
output_counts AS (
  SELECT COUNT(*) AS result_count
  FROM customers c
  JOIN orders o ON c.customer_id = o.customer_id
)
SELECT 
  i.customer_count,
  o.result_count,
  o.result_count - i.customer_count AS unexpected_rows,
  ROUND(o.result_count * 1.0 / i.customer_count, 2) AS fan_out_factor
FROM input_counts i, output_counts o;
```

**2. Check for duplicate keys**
```sql
-- Find duplicate join keys in orders table
SELECT 
  customer_id,
  COUNT(*) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY order_count DESC
LIMIT 10;
```

#### **Common Fan-Out Scenarios:**

**Scenario 1: Multiple Detail Tables**
```sql
-- ❌ Bad: Creates Cartesian product between orders and payments!
SELECT c.*, o.*, p.*
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN payments p ON c.customer_id = p.customer_id;
-- If customer has 5 orders and 3 payments = 15 rows per customer!

-- ✅ Good: Join through order_id instead
SELECT c.*, o.*, p.*
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN payments p ON o.order_id = p.order_id;
-- Or aggregate one of the detail tables first
```

**Scenario 2: Duplicate Dimension Records**
```sql
-- ❌ Product table has duplicates
products:
product_id | name
-----------+--------
101        | Laptop
101        | Laptop  -- Duplicate!

-- Query creates double rows
SELECT o.*, p.name
FROM orders o
JOIN products p ON o.product_id = p.product_id;
-- Every order with product_id 101 appears twice!

-- ✅ Fix: Remove duplicates first
WITH unique_products AS (
  SELECT DISTINCT product_id, name FROM products
)
SELECT o.*, p.name
FROM orders o
JOIN unique_products p ON o.product_id = p.product_id;
```

#### **Cardinality Relationships:**

**1:1 (One-to-One)**
```sql
-- Each customer has exactly one profile
FROM customers c
JOIN customer_profiles p ON c.customer_id = p.customer_id;
-- Input rows = Output rows
```

**1:N (One-to-Many)**
```sql
-- Each customer has many orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id;
-- Output rows = sum of orders per customer
-- Always causes fan-out!
```

**N:M (Many-to-Many)**
```sql
-- Students ↔ Courses (via junction table)
FROM students s
JOIN enrollments e ON s.student_id = e.student_id
JOIN courses c ON e.course_id = c.course_id;
-- Complex fan-out: student with 5 courses = 5 rows
```

#### **Best Practices:**

**1. Always verify row counts**
```sql
SELECT COUNT(*) FROM customers;  -- Before
SELECT COUNT(*) FROM customers c JOIN orders o ...;  -- After
-- Does the difference make sense?
```

**2. Use DISTINCT carefully**
```sql
-- ⚠️ DISTINCT can hide bugs!
SELECT DISTINCT c.customer_id, c.name
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id;
-- Masks the fan-out, but doesn't fix the root cause
```

**3. Aggregate at the right level**
```sql
-- Aggregate in subquery, not in main query
WITH agg AS (
  SELECT customer_id, COUNT(*) AS cnt
  FROM detail_table
  GROUP BY customer_id
)
SELECT c.*, agg.cnt
FROM customers c
LEFT JOIN agg ON c.customer_id = agg.customer_id;
```

**4. Document expected cardinality**
```sql
-- Add comments explaining relationships
SELECT ...
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id  -- 1:N
JOIN order_items oi ON o.order_id = oi.order_id  -- 1:N
-- Expected fan-out: avg 5 orders/customer × 3 items/order = 15x
```

#### **Interview Answer Template:**

1. **Define fan-out:** "More output rows than input rows due to 1:N relationships"
2. **Identify cause:** "Check for duplicate keys, multiple detail tables"
3. **Provide fixes:** "Aggregate in subquery, GROUP BY, or use window functions"
4. **Verify:** "Always compare input/output row counts"

## 🎓 Congratulations - You've Mastered Advanced JOINs & Aggregations!

### 📊 What You've Learned:

✅ **Self JOINs** - Hierarchies, employee-manager, duplicate detection, comparisons
✅ **CROSS JOINs** - Date series, Cartesian products, all combinations
✅ **Anti-JOINs** - NOT EXISTS, NOT IN (with NULL awareness), finding missing data
✅ **ROLLUP** - Hierarchical subtotals, report-friendly aggregations
✅ **CUBE** - Multi-dimensional analysis, all combination subtotals
✅ **GROUPING SETS** - Custom subtotal combinations, performance optimization
✅ **Conditional Aggregates** - CASE WHEN in aggregates, pivot reports, filtered metrics
✅ **JOIN Performance** - Fan-out detection, cardinality awareness, optimization

---

### 🚀 Key Takeaways:

**Self JOIN Pattern:**
```sql
FROM table t1
JOIN table t2 ON t1.key = t2.parent_key
WHERE t1.id != t2.id  -- Avoid self-match
```

**Anti-JOIN (NOT EXISTS) - Preferred:**
```sql
WHERE NOT EXISTS (
  SELECT 1 FROM other_table WHERE key = outer.key
)
-- NULL-safe, performant
```

**NOT IN Trap:**
```sql
-- ❌ Fails with NULL in subquery!
WHERE key NOT IN (SELECT key FROM table)  

-- ✅ Filter NULLs or use NOT EXISTS
WHERE key NOT IN (SELECT key FROM table WHERE key IS NOT NULL)
```

**ROLLUP vs CUBE:**
- **ROLLUP:** Hierarchical (follows column order) - N+1 groupings
- **CUBE:** All combinations - 2^N groupings
- **GROUPING SETS:** Custom combinations - only what you need

**Conditional Aggregates:**
```sql
SUM(CASE WHEN condition THEN value ELSE 0 END)
COUNT(CASE WHEN condition THEN 1 END)
AVG(CASE WHEN condition THEN value END)
```

**Fan-Out Prevention:**
1. Understand cardinality (1:1, 1:N, N:M)
2. Aggregate detail tables in subqueries first
3. Always verify input vs output row counts
4. Use GROUP BY when appropriate

---

### 🎯 Interview Day Checklist:

✅ Can you write a self JOIN for employee-manager hierarchy?
✅ Do you know when to use LEFT JOIN vs NOT EXISTS?
✅ Can you explain the NOT IN NULL trap?
✅ Can you write ROLLUP and CUBE queries?
✅ Do you understand GROUPING() and GROUPING_ID()?
✅ Can you create pivot reports with CASE WHEN?
✅ Can you detect and fix JOIN fan-out?
✅ Do you know how to verify JOIN cardinality?

---

### 📚 Next Steps:

1. **Practice Complex JOINs** - Work with 5+ table queries
2. **Master Window Functions** - Move to Module 3
3. **Optimize Performance** - Learn EXPLAIN plans and indexing
4. **Real Datasets** - Practice on production-sized data
5. **Data Quality** - Build checks using anti-JOINs and aggregates

---

### 💡 Final Interview Tips:

**Clarify requirements:**
- "Is this a 1:N or N:M relationship?"
- "Should the result include customers with no orders?"
- "Do we expect duplicates in the join key?"

**Explain your approach:**
- "I'm using NOT EXISTS instead of NOT IN because..."
- "I'll aggregate first to avoid fan-out"
- "This is a self JOIN to handle the hierarchy"

**Validate assumptions:**
- "Let me check for duplicate keys first"
- "I'll verify the row count matches expectations"
- "Let me ensure there are no NULLs in the join column"

**Discuss alternatives:**
- "We could use CUBE, but GROUPING SETS would be more efficient"
- "LEFT JOIN + NULL check vs NOT EXISTS - both work, I prefer NOT EXISTS for performance"

---

**You're now ready for advanced JOIN and aggregation interviews!** 🎉

Good luck! 🚀

*Pro tip: The difference between mid-level and senior engineers is understanding WHY a query returns unexpected rows. Always think about cardinality!*